In [1]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
    "org.apache.iceberg:iceberg-aws-bundle:1.10.0 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("gold_congestion")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "password123")
    .getOrCreate()
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.taxi")

DataFrame[]

In [2]:
silver = spark.table("lakehouse.taxi.silver_trips")

gold_impact = (
    silver
    .groupBy(
        "pickup_date",
        "pickup_hour",
        "PULocationID",
        "pickup_zone",
        "pickup_borough"
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("speed_mph").alias("avg_speed_mph"),
        F.avg("congestion_surcharge").alias("avg_congestion_surcharge"),
        F.sum(F.when(F.col("congestion_surcharge") > 0, 1).otherwise(0)).alias("congested_trips"),
        F.sum(F.when(F.col("congestion_surcharge") == 0, 1).otherwise(0)).alias("non_congested_trips"),
        F.sum("congestion_surcharge").alias("total_congestion_revenue"),
        F.avg("fare_per_mile").alias("avg_fare_per_mile"),
        F.avg("fare_amount").alias("avg_fare_amount"),
        F.sum("fare_amount").alias("total_fare_revenue"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("duration_minutes").alias("avg_duration_minutes")
    )
    .filter(F.col("pickup_zone").isNotNull())
)

gold_impact.writeTo("lakehouse.taxi.gold_congestion_impact").createOrReplace()

print("Gold congestion impact table created.")
spark.sql("SELECT COUNT(*) AS gold_impact_rows FROM lakehouse.taxi.gold_congestion_impact").show()

spark.sql("""
SELECT
    pickup_date,
    pickup_hour,
    pickup_zone,
    pickup_borough,
    trip_count,
    ROUND(avg_speed_mph, 2) AS avg_speed_mph,
    ROUND(avg_congestion_surcharge, 2) AS avg_congestion_surcharge,
    congested_trips,
    non_congested_trips,
    ROUND(total_congestion_revenue, 2) AS total_congestion_revenue,
    ROUND(avg_fare_per_mile, 2) AS avg_fare_per_mile
FROM lakehouse.taxi.gold_congestion_impact
ORDER BY pickup_date, pickup_hour, pickup_zone
LIMIT 20
""").show(truncate=False)

Gold congestion impact table created.
+----------------+
|gold_impact_rows|
+----------------+
|          189899|
+----------------+

+-----------+-----------+-----------------------------+--------------+----------+-------------+------------------------+---------------+-------------------+------------------------+-----------------+
|pickup_date|pickup_hour|pickup_zone                  |pickup_borough|trip_count|avg_speed_mph|avg_congestion_surcharge|congested_trips|non_congested_trips|total_congestion_revenue|avg_fare_per_mile|
+-----------+-----------+-----------------------------+--------------+----------+-------------+------------------------+---------------+-------------------+------------------------+-----------------+
|2024-12-31 |20         |Clinton East                 |Manhattan     |1         |16.96        |2.5                     |1              |0                  |2.5                     |5.41             |
|2024-12-31 |20         |West Chelsea/Hudson Yards    |Manhattan  

In [3]:
daily_zone = (
    spark.table("lakehouse.taxi.gold_congestion_impact")
    .groupBy("pickup_date", "PULocationID", "pickup_zone", "pickup_borough")
    .agg(
        F.sum("trip_count").alias("daily_trip_count"),
        F.avg("avg_speed_mph").alias("daily_avg_speed_mph"),
        F.sum("total_congestion_revenue").alias("daily_congestion_revenue"),
        F.avg("avg_fare_per_mile").alias("daily_avg_fare_per_mile")
    )
)

speed_rank_window = Window.partitionBy("pickup_date").orderBy(F.col("daily_avg_speed_mph").asc())
revenue_rank_window = Window.partitionBy("pickup_date").orderBy(F.col("daily_congestion_revenue").desc())

gold_zones = (
    daily_zone
    .withColumn("slow_speed_rank", F.row_number().over(speed_rank_window))
    .withColumn("revenue_rank", F.row_number().over(revenue_rank_window))
    .withColumn(
        "zone_category",
        F.when(F.col("slow_speed_rank") <= 5, F.lit("top_5_slowest"))
         .when(F.col("revenue_rank") <= 5, F.lit("top_5_revenue"))
         .otherwise(F.lit("other"))
    )
    .filter((F.col("slow_speed_rank") <= 5) | (F.col("revenue_rank") <= 5))
)

gold_zones.writeTo("lakehouse.taxi.gold_congestion_zones").createOrReplace()

print("Gold congestion zones daily summary created.")
spark.sql("SELECT COUNT(*) AS gold_zone_rows FROM lakehouse.taxi.gold_congestion_zones").show()

spark.sql("""
SELECT
    pickup_date,
    pickup_zone,
    pickup_borough,
    daily_trip_count,
    ROUND(daily_avg_speed_mph, 2) AS daily_avg_speed_mph,
    ROUND(daily_congestion_revenue, 2) AS daily_congestion_revenue,
    slow_speed_rank,
    revenue_rank,
    zone_category
FROM lakehouse.taxi.gold_congestion_zones
ORDER BY pickup_date, slow_speed_rank, revenue_rank
LIMIT 20
""").show(truncate=False)

Gold congestion zones daily summary created.
+--------------+
|gold_zone_rows|
+--------------+
|           590|
+--------------+

+-----------+-----------------------------+--------------+----------------+-------------------+------------------------+---------------+------------+-------------+
|pickup_date|pickup_zone                  |pickup_borough|daily_trip_count|daily_avg_speed_mph|daily_congestion_revenue|slow_speed_rank|revenue_rank|zone_category|
+-----------+-----------------------------+--------------+----------------+-------------------+------------------------+---------------+------------+-------------+
|2024-12-31 |Central Park                 |Manhattan     |1               |6.08               |2.5                     |1              |4           |top_5_slowest|
|2024-12-31 |West Chelsea/Hudson Yards    |Manhattan     |2               |8.11               |5.0                     |2              |1           |top_5_slowest|
|2024-12-31 |Midtown North                |Manhat

In [4]:
spark.sql("""
SELECT
    pickup_date,
    pickup_zone,
    pickup_borough,
    trip_count,
    ROUND(avg_speed_mph, 2) AS avg_speed_mph,
    ROUND(avg_congestion_surcharge, 2) AS avg_congestion_surcharge,
    ROUND(total_congestion_revenue, 2) AS total_congestion_revenue,
    ROUND(avg_fare_per_mile, 2) AS avg_fare_per_mile
FROM lakehouse.taxi.gold_congestion_impact
WHERE pickup_hour = 8
ORDER BY avg_speed_mph ASC
LIMIT 10
""").show(truncate=False)

+-----------+-----------------+--------------+----------+-------------+------------------------+------------------------+-----------------+
|pickup_date|pickup_zone      |pickup_borough|trip_count|avg_speed_mph|avg_congestion_surcharge|total_congestion_revenue|avg_fare_per_mile|
+-----------+-----------------+--------------+----------+-------------+------------------------+------------------------+-----------------+
|2025-01-26 |Parkchester      |Bronx         |1         |0.07         |0.0                     |0.0                     |535.0            |
|2025-02-27 |East Flushing    |Queens        |1         |0.13         |0.0                     |0.0                     |322.5            |
|2025-02-04 |Crotona Park East|Bronx         |1         |0.27         |0.0                     |0.0                     |285.0            |
|2025-01-01 |Jamaica Estates  |Queens        |1         |0.86         |0.0                     |0.0                     |27.5             |
|2025-02-23 |Rego Pa

In [5]:
spark.sql("""
SELECT
    pickup_date,
    ROUND(SUM(total_congestion_revenue), 2) AS daily_total_congestion_revenue
FROM lakehouse.taxi.gold_congestion_impact
GROUP BY pickup_date
ORDER BY pickup_date
""").show(truncate=False)

+-----------+------------------------------+
|pickup_date|daily_total_congestion_revenue|
+-----------+------------------------------+
|2024-12-31 |42.5                          |
|2025-01-01 |161800.0                      |
|2025-01-02 |177792.5                      |
|2025-01-03 |193037.5                      |
|2025-01-04 |208740.0                      |
|2025-01-05 |165000.0                      |
|2025-01-06 |166752.5                      |
|2025-01-07 |207060.0                      |
|2025-01-08 |226012.5                      |
|2025-01-09 |235925.0                      |
|2025-01-10 |221250.0                      |
|2025-01-11 |237232.5                      |
|2025-01-12 |189295.0                      |
|2025-01-13 |198172.5                      |
|2025-01-14 |236197.5                      |
|2025-01-15 |242555.0                      |
|2025-01-16 |259465.0                      |
|2025-01-17 |224822.5                      |
|2025-01-18 |218732.5                      |
|2025-01-1

In [6]:
spark.sql("""
WITH daily_slowest AS (
    SELECT
        pickup_date,
        PULocationID,
        pickup_zone,
        pickup_borough,
        daily_avg_speed_mph,
        ROW_NUMBER() OVER (
            PARTITION BY pickup_date
            ORDER BY daily_avg_speed_mph ASC
        ) AS speed_rank
    FROM (
        SELECT
            pickup_date,
            PULocationID,
            pickup_zone,
            pickup_borough,
            AVG(avg_speed_mph) AS daily_avg_speed_mph
        FROM lakehouse.taxi.gold_congestion_impact
        GROUP BY pickup_date, PULocationID, pickup_zone, pickup_borough
    )
),
top3 AS (
    SELECT *
    FROM daily_slowest
    WHERE speed_rank <= 3
)
SELECT
    g.pickup_date,
    g.pickup_hour,
    g.pickup_zone,
    g.pickup_borough,
    ROUND(g.avg_speed_mph, 2) AS hourly_avg_speed_mph,
    g.trip_count,
    ROUND(g.total_congestion_revenue, 2) AS hourly_congestion_revenue
FROM lakehouse.taxi.gold_congestion_impact g
JOIN top3 t
    ON g.pickup_date = t.pickup_date
   AND g.PULocationID = t.PULocationID
ORDER BY g.pickup_date, t.speed_rank, g.pickup_hour
LIMIT 100
""").show(truncate=False)

+-----------+-----------+-------------------------------+--------------+--------------------+----------+-------------------------+
|pickup_date|pickup_hour|pickup_zone                    |pickup_borough|hourly_avg_speed_mph|trip_count|hourly_congestion_revenue|
+-----------+-----------+-------------------------------+--------------+--------------------+----------+-------------------------+
|2024-12-31 |23         |Central Park                   |Manhattan     |6.08                |1         |2.5                      |
|2024-12-31 |20         |West Chelsea/Hudson Yards      |Manhattan     |2.35                |1         |2.5                      |
|2024-12-31 |23         |West Chelsea/Hudson Yards      |Manhattan     |13.86               |1         |2.5                      |
|2024-12-31 |23         |Midtown North                  |Manhattan     |8.54                |1         |2.5                      |
|2025-01-01 |7          |Jamaica Estates                |Queens        |13.45      

In [8]:
spark.sql("""
SELECT
    pickup_date,
    pickup_zone,
    pickup_borough,
    trip_count,
    ROUND(avg_speed_mph, 2) AS avg_speed_mph,
    ROUND(avg_congestion_surcharge, 2) AS avg_congestion_surcharge,
    ROUND(total_congestion_revenue, 2) AS total_congestion_revenue,
    ROUND(avg_fare_per_mile, 2) AS avg_fare_per_mile
FROM lakehouse.taxi.gold_congestion_impact
WHERE pickup_hour = 8
  AND trip_count >= 20
ORDER BY avg_speed_mph ASC
LIMIT 10
""").show(truncate=False)

+-----------+-----------------------------+--------------+----------+-------------+------------------------+------------------------+-----------------+
|pickup_date|pickup_zone                  |pickup_borough|trip_count|avg_speed_mph|avg_congestion_surcharge|total_congestion_revenue|avg_fare_per_mile|
+-----------+-----------------------------+--------------+----------+-------------+------------------------+------------------------+-----------------+
|2025-02-04 |Sutton Place/Turtle Bay North|Manhattan     |116       |5.87         |2.18                    |252.5                   |12.72            |
|2025-01-23 |Sutton Place/Turtle Bay North|Manhattan     |169       |6.16         |2.06                    |347.5                   |12.23            |
|2025-02-06 |Sutton Place/Turtle Bay North|Manhattan     |116       |6.37         |1.96                    |227.5                   |12.03            |
|2025-02-06 |Upper East Side South        |Manhattan     |338       |6.4          |2.16 